<div style="text-align:center;">
  <img src="https://github.com/MolSSI-Education/iqb-2025/blob/main/images/molssi_main_outline.png?raw=true" style="display: block; margin: 0 auto; max-height:325px;">
</div>

# Analyzing MD trajectories
*This tutorial was created by Levi Naden (Software Scientist) at The Molecular Sciences Software Institue for the MERCURY 26 workshop*


This notebook will walk you through the process of loading data into a specific trajectory analysis tool (MDAnalysis), selecting subgroups of the trajectory, looking at energies for equilibration, separating out correlated samples, generating poses for docking, and then saving those poses for docking-ready input files.


# Set Up

Before going into any details, we'll be installing all the packages we need for this notebook. Specifically for the actual workflow we'll use:
* `numpy`
* `MDAnalysis`
* `chemcompute`
* `pandas`
* `nglview`

In [ ]:
# Load Libraries
import nglview
import numpy as np
import pandas as pd
import MDAnalysis as mda
from chemcompute import chemcompute

## Visualizing Your Simulation

First and foremost, we need our simulation. We're going to pull the simulation from ChemCompute.

The ChemCompute Python module has the ability to load the trajectory and topology directly. We're going to load those in and then work from there. You'll need your simulation ID number from the ChemComptue dashboard.

In [ ]:
cc_id

The actual trajectory can be split into many different sub-trajectories based on which atom groupings you want to capture. If you want to see what trajectories are available to the job, you can use the `list_trajectories` method of the `chemcompute` module.

In [ ]:
chemcompute.list_trajectories(cc_id)

As you can see, there are several trajectories there which we can pull down, with details about what each of them are. We'll be working with the `filtered` simulation for now, which is interesting molecular features as its our simulation less the water. To visualize that, we can use a built in function for ChemCompute called `show_trajectory`

In [ ]:
chemcompute.show_trajectory(cc_id, variant='filtered')

<div class="alert alert-block alert-warning">
<h3>Excercise</h3>
<p>Visualize the solvated trajectory through chemcompute</p>
<p>Questions to consider:
    <ul>
        <li>What why is this simulation more choppy?</li>
        <li>How would you manipulate or get extra data from the simulation generated by this function?</li>
        <li>What is actually the Python <code>object</code>/thing being generated by this function?</li>
</ul>
<p><i>Hint: What is this section called in the notebook?</i></p>


In [ ]:
# Use this box for your answer

## Loading your trajectory into Python

If we want to actually manipulate our trajectory, we need something which is not just the visualization of it, we need it in Python. Fortunatley, `chemcompute` has a function which will do just that called `load_trajectory`. It works the same as `show_trajectory`, but returns a different object.

In [ ]:
u = chemcompute.load_trajectory(cc_id)
u

This object is called a `Universe` and it is the main work object for the package `MDAnalysis`. There are many other packages which do similar things to `MDAnalysis` in reading and parsing trajectories, but this one comes built into ChemCompute's service, so we'll go with that.

We can also see that the `Universe` has a trajectory loaded with it by checking the `trajectory` property

In [ ]:
u.trajectory

I want to take a second and stop to look at what is needed to make a `Universe` object in `MDAnalysis`,and for that, we go to [MDAnalysis's Documentation on `Universe`](https://userguide.mdanalysis.org/stable/universe.html)

Always remember: when in doubt, check the docs.

We can see that `Universe` has a few different ways it can be called:

```python
u = Universe(topology, trajectory)
u = Universe(pdbfile)                       # read atoms and coordinates from PDB or GRO
u = Universe(topology, [traj1, traj2, ...]) # read from a list of trajectories
u = Universe(topology, traj1, traj2, ...)   # read from multiple trajectories
```

And generally takes both something called "Topology" and a  "Trajectory", although the trajectory is optional. Its important to understand all the information that went into a simulation to get out a "Trajectory."

* Forcefield (or Force Field): Parameters and rules for defining the potential energy given a set of atoms under certain conditions
* Topology: The set of atoms, connectivity, and structure chart of all the particles that make up your simulation
* Coordinates and Velocities: The positions and speeds each particle is moving initially. Velocities are often not provided as they can be initilized from the temperature.

Any simulation is then: FF + Topology + Coordinates at minimum.

Why do you think we need the topology in the `Universe`?

## Exploring the Universe

The `Unverse` object is mostly a collection of different **iterators** you can manipulate and has a number of handy built in properties you can access to look at different iterators of the topology. Some helpful ones:
* `.residues`
* `.bonds`
* `.atoms`
and each of these is often both **indexable** and **slicable** in python with their own subselections. Lets look at some below:

In [ ]:
# See the generic, unsliced/indexed residues
u.residues

In [ ]:
# On its own, doesnt do much, but if we index it more carefully...
res_15 = u.residues[15]
res_15

In [ ]:
# And from that, we can get the specific atoms...
res_15_atoms = res_15.atoms
res_15_atoms

In [ ]:
# And then keep going down, this time with a slice...
val_15_every_other_bond = res_15_atoms[::2].bonds
val_15_every_other_bond

We've now made a complex selection where we took every other atom of the valine residue at index 15 of the molecular system and got the set of bonds conencting those atoms. Does this specific group object help us any? I don't know, probably not. But by indexing and slicing, we can get way down to whatever grouping of atoms we want in the topology, including geting some specific data out of that selection.

In [ ]:
val_15_every_other_bond[0].length()

Wait! But what were the units of that!?

Angstroms.

How do you know that?!

[The docs](https://userguide.mdanalysis.org/stable/units.html).

...oh yeah!

<div class="alert alert-block alert-warning">
<h3>Excercise: MDAnalysis Knowledge Check</h3>
<p>Group atoms based on where they are in the system</p>
<p>I want you to use slices and indicies for this exercise to make your different groups. And I want three of them: Protein, Water, Ligand
    <ul>
        <li>You'll need a simulation with water in it</li>
        <li>It will help to look at the methods you can act with for a single data container (atom/residue/etc) to see what you can do with it</li>
        <li>Remember, I want the <i>atoms</i> for each group</li>
        <li>There are multiple, equally correct ways to do this.</li>
        <li>Try by yourself first, then ask for help from neighbors.</li>
        <li>There's an easier way to do this without slices and indices, but practice this for now to get feel for how MDAnalysis structures its data</li>
</ul>
<p><i>Hint: Look at the comment I have in each of the boxes below to see what you may want to do.</i></p>

In [ ]:
# Load a simulation of your system with water

In [ ]:
# See all the possible resiude names.
# Hint, np.unique is not the same as Univers.unique, one may be useful here.

In [ ]:
# Identify the ligand and water residue names, run through the residues and append the indexes to your sorted groups of indices.
# Hint: MDAnalysis calls the index "ix"
# Hint: A chlorine Ion "residue" in this simulation is called "CLA"
water_name
lig_name
chlorine = "CLA"
water_idx = []
lig_idx = []
protein_idx = []

In [ ]:
# Select the residues from the Universe
# Select the atoms
water
ligand
protein

## Getting data from the MDAnalysis Trajectory

So far all we have done is manipulate the initial configuration that was fed into MDAnalysis, what about the trajectory itself? Recall what a simulation Trajectory actually is...

In [ ]:
u.trajectory

In our case, its a simulation of so many frames and so many atoms. What does that mean in pratice? If you wanted to look at the file directly, you could...

In [ ]:
u.trajectory.filename

But since this is a `dcd` file, its not meant to be human readable. Its encoded for space. A trajectory, at its core, is just the coordinates of your system as it evolves with time. Some trajectories can store additional information if its time dependent or sampling dependent (case-by-case application), however, our trajectory is just the position of every particle at every time step. 

Note how I said "particle" and not "atom." The simulation **does not care** what the "atom" is when its running; the simulation is propagating this thing you and I know is an atom, but it knows is a point with coordinates and parameters, through time. The actual relation between the particle index and the atom is stored in the topology, and the simulation (usually) has no need to write out the topology information every step.

That's alot of fancy words to say the "Trajectory" is just an `N x 3 x T` array of floats with `N` being the number of particles, `3` because of the xyz dimensionality, and `T` number of timesteps that were recorded.

And just like everything else in MDAnalysis, a `trajectory` can also be sliced, indxed, and iterated over (your most common action).

In [ ]:
for ts in u.trajectory:
    # As you iterate through a trajectory, MDAnalysis is updating the properties for every other object you can access in real time
    print(f"The center of mass for the system at frame {ts.frame} is {u.atoms.center_of_mass()}")


In [ ]:
# Once out of the iterator, timestep is always reset to zero
u.trajectory.time

Since the `Universe` object is evolving in real time with the trajectory iterator, our selectors we specifed will *also be evolving with the iterator* and we can use those directly.

<div class="alert alert-block alert-warning">
<h3>Displacement of Protein with Time</h3>
<p>How far did the protein's center of mass move with time?</p>
<p>Using your selection tools from above, tell me how far the protein moved each time step
<p><i>Hint:<code>np.linalg.norm(a-b)</code></i></p>

In [ ]:
# Find your reference, iterate, measure. 

## Better Selection Syntax

MDAnalysis does have a built in way to do atom selection in a better way. The function is `select_atoms` and acts on the `Universe`. It uses a human parsable string to interpret what you want. I won't cover everything it can do, but the [Atom Selection docs](https://userguide.mdanalysis.org/stable/selections.html) are very helpful for this.

In [ ]:
#Use select_atoms to get the same thing as our groups from the first exersice

# I'm loading the file here again just to keep a consistent name. Please feel free to use your own.
u_solv = chemcompute.load_trajectory(cc_id, variant='solvated')

water_sel = u_solv.select_atoms("water")
ligand_sel = u_solv.select_atoms("resname 13U")
protein_sel = u_solv.select_atoms("protein")

for manual, select in zip([water, ligand, protein], [water_sel, ligand_sel, protein_sel]):
    print(np.array_equal(manual.indices, select.indices))

The selection language can also get much more complicated and powerful

In [ ]:
# This will give you the ligand again. Not the most helpful here however...
ligand_select = "segid AO2 and record_type ATOM and not resname CLA"
ligand = u.select_atoms(ligand_select)

# Sometimes PDB records can get... complicated. This one Dr. Naden has used to separate ligand from a solvated dimer structure 7LME
ligand_select_sample = "segid A and record_type HETATM and not resname HOH"
ligand_sample = u.select_atoms(ligand_select)

[Link to the 7LME dimer structure](https://www.rcsb.org/structure/7LME) for reference.

And we can use our existing selections to make much more helpful selections. For instance, all the atoms within a certain distance of the ligand:

In [ ]:
# Find and atoms within a certain distance from the ligand
active_site = u.select_atoms("around 3.5 group ligand",
                             periodic=False, # selection that prevents "geometric" selection operators from wrapping around the PBC 
                             ligand=ligand)  # Uses generic select_name=object as kwargs
print(active_site)

<div class="alert alert-block alert-warning">
<h3>Chaining and Compound Selections</h3>
<p>Using the selection language, find the following two things:</p>
        <ul>
            <li>The backbone atoms of the binding site</li>
            <li>If there are any waters inside the active site (this is a bit fuzzy)</li>
            <li>BOUNUS: If there were <i>ever</i> any waters in the binding site...</li>
<p><i>Hint: Check the atom selection docs and the selection language function that will help you the most is "sphzone."</i></p>

## Incorperating data

Looking over the trajectory is all well and good, but we also need to bring in actual data to check things over. We're going to load in our data from our run, and then look over some of the data.

In [ ]:
# Download the data (as a CSV) from ChemCompute, then load the data into a Pandas DataFrame
energies = pd.read_csv(chemcompute.downloadEnergy(cc_id))
energies

In [ ]:
energies.plot(x="Time", y="Total Energy", kind="line")